# 2026 World Cup predictor

This notebook builds a light Elo-style model on historical FIFA World Cup matches (1930-2022) and layers in FIFA's October 2022 rankings as a safety net for teams with limited World Cup history. Use it to compare win/draw/loss odds for hypothetical 2026 fixtures and to estimate group standings.

What you can do:
- Inspect the learned ratings and the teams most likely to qualify (top 48 by FIFA ranking)
- Call `predict_match(team_a, team_b, host=...)` to get win/draw/loss probabilities
- Use `project_group([...])` to compare expected points for a 4-team group


In [ ]:

import math
import os
import numpy as np
import pandas as pd

DATA_DIR = "Data"
MATCH_PATH = os.path.join(DATA_DIR, "matches_1930_2022.csv")
RANK_PATH = os.path.join(DATA_DIR, "fifa_ranking_2022-10-06.csv")

matches = pd.read_csv(MATCH_PATH)
fifa_rank = pd.read_csv(RANK_PATH)

matches["Date"] = pd.to_datetime(matches["Date"])
matches = matches.sort_values("Date").reset_index(drop=True)
matches["result"] = np.where(
    matches["home_score"] > matches["away_score"],
    1.0,
    np.where(matches["home_score"] < matches["away_score"], 0.0, 0.5),
)

draw_rate = float((matches["result"] == 0.5).mean())
rank_points = fifa_rank.set_index("team")["points"].to_dict()
rank_min, rank_max = min(rank_points.values()), max(rank_points.values())

print(f"Loaded {len(matches)} World Cup matches across {matches['Year'].nunique()} tournaments.")
print(f"Observed draw rate: {draw_rate:.2%}")


## Train Elo-style ratings

Older matches count less (exponential decay), goal margin nudges the rating update, and a small home advantage is applied because World Cup matches are not always on neutral ground.


In [ ]:

def fallback_rating(team: str, base: float = 1500.0) -> float:
    pts = rank_points.get(team)
    if pts is None:
        return base
    return float(np.interp(pts, (rank_min, rank_max), (1400.0, 1900.0)))

def train_elo(
    df: pd.DataFrame,
    base_rating: float = 1500.0,
    k_base: float = 40.0,
    decay_years: float = 20.0,
    home_adv: float = 60.0,
) -> dict:
    ratings: dict[str, float] = {}
    current_year = int(df["Date"].dt.year.max())
    for row in df.itertuples():
        ht, at = row.home_team, row.away_team
        ratings.setdefault(ht, base_rating)
        ratings.setdefault(at, base_rating)

        ra, rb = ratings[ht], ratings[at]
        year = row.Date.year
        k = k_base * math.exp(-(current_year - year) / decay_years) + 10.0
        margin = max(1.0, math.log1p(abs(row.home_score - row.away_score)))

        exp_home = 1.0 / (1.0 + 10 ** (-(ra + home_adv - rb) / 400.0))
        result_home = 1.0 if row.home_score > row.away_score else 0.5 if row.home_score == row.away_score else 0.0

        delta = k * margin
        ratings[ht] += delta * (result_home - exp_home)
        ratings[at] += delta * ((1.0 - result_home) - (1.0 - exp_home))

    mean_rating = float(np.mean(list(ratings.values())))
    shift = base_rating - mean_rating
    return {team: rating + shift for team, rating in ratings.items()}

elo_ratings = train_elo(matches)
rating_table = pd.DataFrame(
    sorted(elo_ratings.items(), key=lambda x: x[1], reverse=True),
    columns=["team", "rating"],
)
rating_table.head(12)


## Match prediction helpers

`predict_match(team_a, team_b, host=...)` returns win/draw/loss probabilities. If one of the teams is hosting (USA, Canada, or Mexico in 2026), pass it as `host` to give them a bump.


In [ ]:

HOST_BOOST = 70.0  # rating points worth of home-field advantage

def rating_for(team: str) -> float:
    return float(elo_ratings.get(team, fallback_rating(team)))

def draw_probability(rating_diff: float, base_draw: float = draw_rate, draw_scale: float = 450.0) -> float:
    return float(base_draw * math.exp(-abs(rating_diff) / draw_scale))

def predict_match(team_a: str, team_b: str, host: str | None = None) -> dict:
    ra, rb = rating_for(team_a), rating_for(team_b)
    boost_a = HOST_BOOST if host == team_a else 0.0
    boost_b = HOST_BOOST if host == team_b else 0.0

    rating_diff = (ra + boost_a) - (rb + boost_b)
    expected_a = 1.0 / (1.0 + 10 ** (-rating_diff / 400.0))

    p_draw = draw_probability(rating_diff)
    p_a = expected_a * (1.0 - p_draw)
    p_b = (1.0 - expected_a) * (1.0 - p_draw)

    return {
        "team_a": team_a,
        "team_b": team_b,
        "team_a_win": p_a,
        "draw": p_draw,
        "team_b_win": p_b,
        "rating_a": ra,
        "rating_b": rb,
        "rating_diff": rating_diff,
        "host": host or "neutral",
    }

def pretty_print(pred: dict) -> None:
    print(
        f"{pred['team_a']} vs {pred['team_b']} ({pred['host']}) -> "
        f"{pred['team_a_win']:.1%} / {pred['draw']:.1%} / {pred['team_b_win']:.1%}"
    )


In [ ]:

example_fixtures = [
    ("Argentina", "France", None),
    ("United States", "Germany", "United States"),
    ("Mexico", "Netherlands", "Mexico"),
    ("Canada", "Japan", "Canada"),
]
for a, b, host in example_fixtures:
    pretty_print(predict_match(a, b, host=host))


## Quick 2026 group projection

Build a 4-team group and compare expected points using the win/draw probabilities above (each pair plays once). Top 48 by FIFA ranking form a reasonable 2026 qualifying pool.


In [ ]:

team_pool_2026 = fifa_rank.sort_values("points", ascending=False).head(48)["team"].tolist()
print("Top seeds likely for 2026:", team_pool_2026[:12])

def expected_points(team_a: str, team_b: str, host: str | None = None) -> tuple[float, float]:
    pred = predict_match(team_a, team_b, host=host if host in (team_a, team_b) else None)
    pts_a = 3.0 * pred["team_a_win"] + pred["draw"]
    pts_b = 3.0 * pred["team_b_win"] + pred["draw"]
    return pts_a, pts_b

def project_group(group: list[str], host: str | None = None) -> pd.DataFrame:
    totals = {team: 0.0 for team in group}
    for i in range(len(group)):
        for j in range(i + 1, len(group)):
            a, b = group[i], group[j]
            pts_a, pts_b = expected_points(a, b, host=host)
            totals[a] += pts_a
            totals[b] += pts_b
    table = pd.DataFrame(
        [{"team": t, "expected_points": totals[t]} for t in group]
    ).sort_values("expected_points", ascending=False)
    return table.reset_index(drop=True)

group_a = ["United States", "Italy", "Nigeria", "Australia"]
project_group(group_a, host="United States")


## Monte Carlo: predict the 2026 winner

This is a rough knockout simulation for a 48-team field. The top 16 rated teams (by our Elo/fallback rating) get a bye to the round of 32; the other 32 play a preliminary round. Hosts (USA, Mexico, Canada) get a home boost in every match. Change `n_iter` or swap in your own `team_pool` to try different scenarios.


In [ ]:

HOSTS_2026 = {"United States", "Mexico", "Canada"}

def simulate_match(team_a: str, team_b: str, rng: np.random.Generator) -> str:
    host = team_a if team_a in HOSTS_2026 else team_b if team_b in HOSTS_2026 else None
    pred = predict_match(team_a, team_b, host=host)
    roll = rng.random()
    if roll < pred["team_a_win"]:
        return team_a
    if roll < pred["team_a_win"] + pred["draw"]:
        strength_a = pred["team_a_win"]
        strength_b = pred["team_b_win"]
        total = max(1e-9, strength_a + strength_b)
        return team_a if rng.random() < strength_a / total else team_b
    return team_b

def pair_and_play(teams: list[str], rng: np.random.Generator) -> list[str]:
    rng.shuffle(teams)
    winners = []
    for i in range(0, len(teams), 2):
        winners.append(simulate_match(teams[i], teams[i + 1], rng))
    return winners

def simulate_tournament(team_pool: list[str], n_iter: int = 3000, random_state: int = 7) -> pd.DataFrame:
    seeded = (
        pd.DataFrame(
            [{"team": t, "rating": rating_for(t)} for t in team_pool]
        )
        .sort_values("rating", ascending=False)
        .reset_index(drop=True)
    )
    top16 = seeded.head(16)["team"].tolist()
    prelim_field = seeded["team"].tolist()[16:]

    rng = np.random.default_rng(random_state)
    counts: dict[str, int] = {}
    for _ in range(n_iter):
        prelim_winners = pair_and_play(prelim_field.copy(), rng)
        round_of_32 = top16 + prelim_winners
        rng.shuffle(round_of_32)

        current = round_of_32
        while len(current) > 1:
            current = pair_and_play(current, rng)
        champion = current[0]
        counts[champion] = counts.get(champion, 0) + 1

    table = pd.DataFrame(
        [{"team": t, "title_prob": wins / n_iter} for t, wins in counts.items()]
    ).sort_values("title_prob", ascending=False)
    return table.reset_index(drop=True)

winners_2026 = simulate_tournament(team_pool_2026, n_iter=3000)
winners_2026.head(12)
